# Notebook 2 — Embed Cleaned Maryland Polyvore Dataset

Runs every `top.jpg` and `bottom.jpg` in the Cleaned-Maryland-Dataset through
the exported FashionCLIP INT8 ONNX model and saves aligned embedding arrays.

**Input:**
- `fashion_clip_vision_int8.onnx` (from Notebook 1)
- `Cleaned-Maryland-Dataset/` folder (12,505 outfit subfolders, each with top.jpg + bottom.jpg)

**Output:**
- `outfit_ids.npy` — string array of outfit folder names, shape (N,)
- `top_embeddings.npy` — float32 array, shape (N, 512)
- `bottom_embeddings.npy` — float32 array, shape (N, 512)

All three arrays are aligned by index — outfit_ids[i] corresponds to top_embeddings[i] and bottom_embeddings[i].

**Run on:** Kaggle GPU or Colab T4. GPU not required but speeds up batching.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q onnxruntime Pillow numpy tqdm

In [ ]:
# ── 2. Configure paths ────────────────────────────────────────────────────────
# Update these paths to match your environment.
#
# Kaggle:
#   DATASET_DIR = "/kaggle/input/cleaned-maryland-polyvore/Cleaned-Maryland-Dataset"
#   MODEL_PATH  = "/kaggle/input/fashion-clip-onnx/fashion_clip_vision_int8.onnx"
#
# Colab (Google Drive):
#   from google.colab import drive
#   drive.mount('/content/drive')
#   DATASET_DIR = "/content/drive/MyDrive/Cleaned-Maryland-Dataset"
#   MODEL_PATH  = "/content/drive/MyDrive/fashion_clip_vision_int8.onnx"

DATASET_DIR = "/kaggle/input/cleaned-maryland-polyvore/Cleaned-Maryland-Dataset"
MODEL_PATH  = "/kaggle/input/fashion-clip-onnx/fashion_clip_vision_int8.onnx"
BATCH_SIZE  = 64   # reduce to 32 if you hit memory issues
OUTPUT_DIR  = "/kaggle/working"

import os
assert os.path.isdir(DATASET_DIR), f"Dataset not found: {DATASET_DIR}"
assert os.path.isfile(MODEL_PATH), f"Model not found: {MODEL_PATH}"
print("Paths OK")

In [ ]:
import numpy as np
from PIL import Image
import onnxruntime as ort
from tqdm import tqdm
from pathlib import Path

# CLIP normalisation constants (must match export_fashion_clip.ipynb)
CLIP_MEAN = np.array([0.48145466, 0.4578275,  0.40821073], dtype=np.float32)
CLIP_STD  = np.array([0.26862954, 0.26130258, 0.27577711], dtype=np.float32)

def preprocess_image(path: str) -> np.ndarray:
    """Load and preprocess one image to CLIP input format."""
    img = Image.open(path).convert("RGB").resize((224, 224), Image.BICUBIC)
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = (arr - CLIP_MEAN) / CLIP_STD
    return arr.transpose(2, 0, 1)  # (3, 224, 224)

print("Preprocessing function ready")

In [ ]:
# ── 3. Discover all valid outfit folders ─────────────────────────────────────
dataset_path = Path(DATASET_DIR)

outfit_dirs = sorted([
    d for d in dataset_path.iterdir()
    if d.is_dir()
    and (d / "top.jpg").exists()
    and (d / "bottom.jpg").exists()
])

print(f"Found {len(outfit_dirs):,} valid outfit folders")

In [ ]:
# ── 4. Load FashionCLIP ONNX model ────────────────────────────────────────────
sess_options = ort.SessionOptions()
sess_options.inter_op_num_threads = 4
sess_options.intra_op_num_threads = 4

sess = ort.InferenceSession(
    MODEL_PATH,
    sess_options=sess_options,
    providers=["CPUExecutionProvider"],
)
print("ONNX model loaded")
print(f"Input:  {sess.get_inputs()[0].name} — shape {sess.get_inputs()[0].shape}")
print(f"Output: {sess.get_outputs()[0].name} — shape {sess.get_outputs()[0].shape}")

In [ ]:
# ── 5. Batch embed all outfits ────────────────────────────────────────────────
def embed_batch(paths):
    """Embed a batch of image paths. Returns (B, 512) float32 array."""
    batch = np.stack([preprocess_image(p) for p in paths])  # (B, 3, 224, 224)
    return sess.run(["embedding"], {"pixel_values": batch})[0]  # (B, 512)

outfit_ids      = []
top_embeddings  = []
bottom_embeddings = []
failed          = []

for i in tqdm(range(0, len(outfit_dirs), BATCH_SIZE), desc="Embedding"):
    batch_dirs = outfit_dirs[i : i + BATCH_SIZE]
    top_paths    = [str(d / "top.jpg")    for d in batch_dirs]
    bottom_paths = [str(d / "bottom.jpg") for d in batch_dirs]

    try:
        top_embs    = embed_batch(top_paths)
        bottom_embs = embed_batch(bottom_paths)
    except Exception as e:
        # Fall back to one-by-one to isolate bad images
        top_embs    = []
        bottom_embs = []
        for t, b, d in zip(top_paths, bottom_paths, batch_dirs):
            try:
                top_embs.append(embed_batch([t])[0])
                bottom_embs.append(embed_batch([b])[0])
            except:
                failed.append(d.name)
                continue
        if not top_embs:
            continue
        top_embs    = np.stack(top_embs)
        bottom_embs = np.stack(bottom_embs)
        batch_dirs  = [d for d in batch_dirs if d.name not in failed]

    for j, d in enumerate(batch_dirs):
        outfit_ids.append(d.name)
        top_embeddings.append(top_embs[j])
        bottom_embeddings.append(bottom_embs[j])

print(f"\nEmbedded: {len(outfit_ids):,} outfits")
if failed:
    print(f"Skipped {len(failed)} corrupt outfits: {failed[:10]}")

In [ ]:
# ── 6. Save aligned arrays ────────────────────────────────────────────────────
outfit_ids_arr      = np.array(outfit_ids)
top_embeddings_arr  = np.stack(top_embeddings).astype(np.float32)
bottom_embeddings_arr = np.stack(bottom_embeddings).astype(np.float32)

print(f"outfit_ids:       {outfit_ids_arr.shape}")
print(f"top_embeddings:   {top_embeddings_arr.shape}")
print(f"bottom_embeddings:{bottom_embeddings_arr.shape}")

np.save(f"{OUTPUT_DIR}/outfit_ids.npy",       outfit_ids_arr)
np.save(f"{OUTPUT_DIR}/top_embeddings.npy",   top_embeddings_arr)
np.save(f"{OUTPUT_DIR}/bottom_embeddings.npy", bottom_embeddings_arr)

print("Saved: outfit_ids.npy, top_embeddings.npy, bottom_embeddings.npy")

In [ ]:
# ── 7. Sanity checks ──────────────────────────────────────────────────────────
# 7a. Same outfit: top and bottom should have moderate-to-high similarity
# 7b. Random pair: should have lower similarity on average

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

same_sims = [
    cosine_sim(top_embeddings_arr[i], bottom_embeddings_arr[i])
    for i in range(min(500, len(outfit_ids_arr)))
]

rng = np.random.default_rng(42)
rand_idx = rng.integers(0, len(outfit_ids_arr), size=(500, 2))
rand_sims = [
    cosine_sim(top_embeddings_arr[i], bottom_embeddings_arr[j])
    for i, j in rand_idx if i != j
]

print(f"Same-outfit cosine similarity:   mean={np.mean(same_sims):.3f}  std={np.std(same_sims):.3f}")
print(f"Random-pair cosine similarity:   mean={np.mean(rand_sims):.3f}  std={np.std(rand_sims):.3f}")
print()
print("Same-outfit similarity should be noticeably higher than random pairs.")
print("If they're similar, something went wrong with embeddings — check preprocessing.")

In [ ]:
# ── 8. Download (Colab) or confirm output (Kaggle) ────────────────────────────
try:
    from google.colab import files
    for fname in ["outfit_ids.npy", "top_embeddings.npy", "bottom_embeddings.npy"]:
        files.download(f"{OUTPUT_DIR}/{fname}")
except ImportError:
    print("Kaggle: save the output dataset so train_compatibility_mlp.ipynb can use it.")
    print("Go to: Notebook → Data → Output → Add as dataset input in the next notebook.")
    for fname in ["outfit_ids.npy", "top_embeddings.npy", "bottom_embeddings.npy"]:
        path = f"{OUTPUT_DIR}/{fname}"
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {fname}: {size_mb:.1f} MB")